# 01c: field-scale scene generator, step by step

`src/01c_scene_generator.py` makes the synthetic training data. For every seed it:

1. checks the fixed grid of the preset against the physics (cell size, time window),
2. draws a random underground scene (soil, layers, targets, rocks) from the seed,
3. writes the gprMax input file `scene.in` and the ground truth `scene.json`,
4. runs gprMax: one A-scan per antenna position, merged into the B-scan `scene_merged.h5`,
5. runs a one-trace reference model of the same soil without targets (`reference.h5`),
6. checks the predicted apex time of every target against the simulated data (`run.json`).

This notebook walks through those steps for one seed and then summarises the whole batch. It reads the outputs that the batch already wrote to `data/sim/scenes/<preset>/seed<N>/`, so it does not run gprMax and finishes in seconds.

Run it in the `gprMax-v4` environment (or any Python with numpy, scipy, h5py, matplotlib and pyyaml).

In [ ]:
import importlib
import json
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

gen = importlib.import_module("01c_scene_generator")
loader = importlib.import_module("02a_gprmax_loader")
proc = importlib.import_module("02b_preprocessing")

PRESET = "field_400mhz"
SEED = 0
config = gen.load_config()
SCENES = ROOT / "data" / "sim" / "scenes" / PRESET
SCENE_DIR = SCENES / f"seed{SEED}"
print("repository:", ROOT)
print("scene folder:", SCENE_DIR, "(exists)" if SCENE_DIR.exists() else "(missing: run the batch first)")

## Step 1: one fixed grid per preset, checked against the physics

All scenes of a preset share one grid, so every B-scan has the same time step, number of samples and number of traces. The script checks that grid against the worst case the preset allows:

- **Cell size.** The slowest wave travels at $v_{min} = c/\sqrt{\varepsilon_{r,max}}$. The highest frequency with real energy in a Ricker pulse is taken as $f_{max} = 2.5 \times f_c$. The shortest wavelength is $\lambda_{min} = v_{min}/f_{max}$, and the cell must be at most $\lambda_{min}/10$, or the simulated wave gets distorted (numerical dispersion).
- **Time window.** Long enough for the echo from the deepest possible target bottom, at the slowest velocity, plus a 25% margin and the delay of the Ricker peak ($\sqrt{2}/f_c$).
- **Trace step and Tx-Rx offset** must be whole numbers of cells, because gprMax places antennas on cell nodes.

In [ ]:
spec = config["presets"][PRESET]
bound = gen.preset_bound(PRESET, config)
print(gen.format_grid(f"{PRESET} bound", bound))
layout = gen.fixed_layout(PRESET, config)
print()
print(f"fixed grid used by every scene: dx {layout['dx'] * 1000:.2f} mm, "
      f"window {layout['time_window_s'] * 1e9:.1f} ns, "
      f"domain {layout['domain_x']:.2f} x {layout['domain_y']:.2f} m ({layout['nx']} x {layout['ny']} cells), "
      f"{layout['n_traces']} traces every {layout['trace_step_m'] * 100:.0f} cm, "
      f"Tx-Rx offset {layout['tx_rx_offset_m']:.4f} m")
print(f"trace step = {layout['trace_step_m'] / layout['dx']:.2f} cells, "
      f"offset = {layout['tx_rx_offset_m'] / layout['dx']:.2f} cells")

## Step 2: a random scene from the seed

The seed starts a random number generator, so the same preset and seed always give the same scene. The scene has:

- a soil half-space and 0 to 2 horizontal layers, each with a random relative permittivity $\varepsilon_r$ and conductivity $\sigma$,
- 1 to 3 targets: a metal pipe (perfect conductor), a PVC pipe (plastic shell filled with air) or an air void, at a random position, depth and radius, never overlapping,
- 0 to 5 small rocks as clutter.

In [ ]:
scene = gen.sample_scene(PRESET, SEED, config)
assert scene == gen.sample_scene(PRESET, SEED, config), "same seed must give the same scene"

def describe(scene):
    print(f"seed {scene['seed']}: surface at y = {scene['surface_y']:.3f} m")
    for band in reversed(scene["column"]):
        print(f"  soil {band['name']:<10} eps_r {band['eps_r']:5.2f}  sigma {band['sigma_s_m'] * 1000:5.2f} mS/m  "
              f"depth {scene['surface_y'] - band['y1']:.2f} to {scene['surface_y'] - band['y0']:.2f} m")
    for t in scene["targets"]:
        print(f"  {t['name']}: {t['type']:<5} x {t['x']:.2f} m, top {t['top_depth_m']:.2f} m, radius {t['radius_m'] * 100:.1f} cm")
    for r in scene["rocks"]:
        print(f"  {r['name']}: rock  x {r['x']:.2f} m, top {r['top_depth_m']:.2f} m, eps_r {r['eps_r']:.1f}")

describe(scene)

### The model as gprMax sees it

The cross-section below is the 2D model: soil bands coloured by permittivity, targets and rocks as circles, the absorbing boundary (PML) as the hatched frame, and the antenna midpoints as ticks along the surface.

In [ ]:
TYPE_COLOUR = {"metal": "#444444", "pvc": "#1f77b4", "air": "#ffffff", "rock": "#a0522d"}

def draw_model(scene, ax):
    import matplotlib.patches as mp
    surface = scene["surface_y"]
    eps_all = [b["eps_r"] for b in scene["column"]]
    cmap = plt.get_cmap("YlOrBr")
    for band in scene["column"]:
        level = (band["eps_r"] - 4) / (12 - 4)
        ax.add_patch(mp.Rectangle((0, band["y0"]), scene["domain_x"], band["y1"] - band["y0"],
                                  color=cmap(0.15 + 0.7 * level), lw=0))
        ax.text(0.05, (band["y0"] + band["y1"]) / 2, f"eps_r {band['eps_r']:.1f}", va="center", fontsize=8)
    ax.add_patch(mp.Rectangle((0, surface), scene["domain_x"], scene["domain_y"] - surface, color="#e8f4fb", lw=0))
    for item in scene["targets"] + scene["rocks"]:
        kind = item["type"]
        ax.add_patch(mp.Circle((item["x"], item["centre_y"]), item["radius_m"],
                               facecolor=TYPE_COLOUR[kind], edgecolor="k", lw=0.8))
        if kind != "rock":
            ax.annotate(f"{item['name']} ({kind})", (item["x"], item["centre_y"] - item["radius_m"]),
                        xytext=(0, -12), textcoords="offset points", ha="center", fontsize=8)
    pml = scene["pml_m"]
    for rect in [(0, 0, pml, scene["domain_y"]), (scene["domain_x"] - pml, 0, pml, scene["domain_y"]),
                 (0, 0, scene["domain_x"], pml), (0, scene["domain_y"] - pml, scene["domain_x"], pml)]:
        ax.add_patch(mp.Rectangle(rect[:2], rect[2], rect[3], fill=False, hatch="///", lw=0, color="0.6"))
    mids = scene["midpoint_x0"] + scene["trace_step_m"] * np.arange(scene["n_traces"])
    ax.plot(mids, np.full(mids.size, scene["antenna_y"]), "|", color="crimson", ms=6, label="antenna midpoints")
    ax.set_xlim(0, scene["domain_x"])
    ax.set_ylim(0, scene["domain_y"])
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.legend(loc="lower right", fontsize=8)

fig, ax = plt.subplots(figsize=(12, 5.5))
draw_model(scene, ax)
ax.set_title(f"{PRESET} seed {SEED}: 2D model ({scene['nx']} x {scene['ny']} cells of {scene['dx'] * 1000:.2f} mm)")
plt.show()

## Step 3: the gprMax input file

`write_model` turns the scene dict into gprMax commands: domain, cell size, time window, materials, the Ricker source and receiver, the step between traces, then the geometry (soil boxes first, then rocks and targets on top). Here it is written to a temporary folder, so the batch outputs are not touched.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    text = gen.write_model(scene, Path(tmp) / "scene.in").read_text()
print(text)

## Step 4: ground truth and predicted hyperbolas

Because the scene is known, the arrival time of each target's echo can be predicted for every antenna position. `hyperbola_time` follows the ray from the transmitter down to the near side of the cylinder and back up to the receiver, through the air gap and every soil layer (with refraction at each interface), and adds the Ricker delay. Plotted against antenna position it is the hyperbola that appears in the B-scan; its top is the **apex**, above the target.

A target counts as **visible** only if its apex arrives at least one period after the **direct wave** (the pulse that travels straight from transmitter to receiver through the air). Earlier echoes merge with the direct wave and cannot be labelled.

In [ ]:
truth = gen.ground_truth(scene)
period_ns = 1e9 / scene["frequency_hz"]
direct_ns = truth["direct_wave_time_s"] * 1e9

fig, ax = plt.subplots(figsize=(10, 4.5))
for t in truth["targets"]:
    x = np.asarray(t["hyperbola_x_m"])
    ax.plot(x, np.asarray(t["hyperbola_t_s"]) * 1e9, label=f"{t['name']} ({t['type']}), apex {t['apex_time_s'] * 1e9:.2f} ns")
    ax.plot(t["x"], t["apex_time_s"] * 1e9, "k.")
ax.axhline(direct_ns, color="crimson", ls="--", label=f"direct wave {direct_ns:.2f} ns")
ax.axhspan(direct_ns, direct_ns + period_ns, color="crimson", alpha=0.1, label="less than one period after it: not visible")
ax.invert_yaxis()
ax.set_xlabel("antenna midpoint [m]")
ax.set_ylabel("two-way time [ns]")
ax.set_ylim(scene["time_window_s"] * 1e9, 0)
ax.legend(fontsize=8)
ax.set_title("Predicted echo time per target")
plt.show()

for t in truth["targets"]:
    print(f"{t['name']}: visible={t['visible']}" + ("" if t["visible"] else f" ({t['visibility_reason']})"))

## Step 5: the simulated B-scan and the reference trace

gprMax simulated one A-scan (one trace) per antenna position and the script merged them into `scene_merged.h5`. The raw B-scan is dominated by the **direct wave** near the top. The **reference model** is the same soil without targets or rocks, run once: because the soil is laterally uniform, its single trace equals the direct wave plus the layer reflections on every trace. Subtracting it leaves only what the targets and rocks add.

In [ ]:
merged = SCENE_DIR / "scene_merged.h5"
reference_path = SCENE_DIR / "reference.h5"
if not merged.exists():
    raise FileNotFoundError(f"{merged} is missing: run the batch for this seed first")

saved = json.loads((SCENE_DIR / "scene.json").read_text())
assert saved["seed"] == SEED and len(saved["targets"]) == len(scene["targets"]), "scene.json does not match the sampled scene"

scan = loader.load_gprmax(merged)
reference = loader.load_gprmax(reference_path)
difference = gen.subtract_reference(scan, reference)
time_ns = np.arange(scan.data.shape[0]) * scan.dt * 1e9
x = np.asarray(truth["targets"][0]["hyperbola_x_m"])
print(f"B-scan: {scan.data.shape[0]} samples x {scan.data.shape[1]} traces, dt = {scan.dt * 1e12:.2f} ps")

def show(ax, data, title, curves=True, shift_ns=None):
    limit = np.percentile(np.abs(data), 99) or 1.0
    ax.imshow(data, extent=[x[0], x[-1], time_ns[-1], time_ns[0]], aspect="auto", cmap="seismic",
              vmin=-limit, vmax=limit, interpolation="nearest")
    if curves:
        for t in truth["targets"]:
            curve = np.asarray(t["hyperbola_t_s"]) * 1e9
            if shift_ns is not None:
                curve = curve - shift_ns
            ax.plot(x, curve, lw=1, ls="--", label=t["type"])
        ax.legend(fontsize=7, loc="lower right")
    ax.set_ylim(time_ns[-1], 0)
    ax.set_xlim(x[0], x[-1])
    ax.set_title(title)
    ax.set_xlabel("antenna midpoint [m]")

fig = plt.figure(figsize=(14, 5))
grid = fig.add_gridspec(1, 4, width_ratios=[4, 1, 4, 0.1])
ax0, ax1, ax2 = fig.add_subplot(grid[0]), fig.add_subplot(grid[1]), fig.add_subplot(grid[2])
show(ax0, scan.data, "raw B-scan", curves=False)
ax0.set_ylabel("time [ns]")
ax1.plot(reference.data[:, 0], time_ns, color="k", lw=0.8)
ax1.set_ylim(time_ns[-1], 0)
ax1.set_title("reference trace")
ax1.set_yticks([])
show(ax2, difference, "raw minus reference, with predicted hyperbolas")
plt.tight_layout()
plt.show()

## Step 6: the standard processing chain

The training images will not use the reference (real surveys do not have one). They use the same processing as real data, from `02b_preprocessing.py`: time zero, dewow, background removal (mean trace), band-pass around the centre frequency, and AGC gain. Dewow and AGC windows are three centre periods. Time zero shifts every trace so that the direct wave starts at 0, so the predicted curves are shifted by the same amount.

In [ ]:
dewow_ns, agc_ns = gen.processing_windows_ns(scene["frequency_hz"])
shifted, picks = proc.time_zero(scan.data, scan.dt)
steps = {"time zero": shifted}
steps["dewow"] = proc.dewow(steps["time zero"], scan.dt, dewow_ns)
steps["background removal"] = proc.remove_background(steps["dewow"])
low, high = proc.default_band(scene["frequency_hz"], 0.5 / scan.dt)
steps["band-pass"] = proc.bandpass(steps["background removal"], scan.dt, low, high)
steps["AGC gain"] = proc.gain_agc(steps["band-pass"], scan.dt, agc_ns)
shift_ns = np.asarray(picks)[: x.size] * scan.dt * 1e9
print(f"dewow {dewow_ns:.1f} ns, band-pass {low / 1e6:.0f}-{high / 1e6:.0f} MHz, AGC {agc_ns:.1f} ns")

fig, axes = plt.subplots(1, len(steps), figsize=(18, 4.5), sharey=True)
for ax, (name, data) in zip(axes, steps.items()):
    show(ax, data, name, curves=(name == "AGC gain"), shift_ns=shift_ns)
axes[0].set_ylabel("time after time zero [ns]")
plt.tight_layout()
plt.show()

## Step 7: the apex check

To check the prediction, the script takes the trace closest to each target, computes the envelope of the reference-subtracted data, and picks its maximum within half a period of the predicted apex time. A visible target passes if the pick is within a quarter period ($0.63$ ns at 400 MHz). A pick on the edge of the search window means no peak was found inside it.

In [ ]:
rows = gen.apex_check(scan, scene, truth, reference)
print(f"{'target':<10}{'type':<7}{'visible':<9}{'predicted ns':>13}{'picked ns':>11}{'error ns':>10}  status")
for r in rows:
    ok = abs(r["error_ns"]) <= r["quarter_period_ns"]
    status = "ok" if ok else ("window edge" if r["on_window_edge"] else "outside quarter period")
    print(f"{r['name']:<10}{r['type']:<7}{str(r['visible']):<9}{r['predicted_ns']:>13.2f}{r['picked_ns']:>11.2f}{r['error_ns']:>10.2f}  {status}")

## Step 8: the whole batch

Every seed folder holds `scene.json` (the scene and its ground truth) and `run.json` (wall time and apex check). The figures below summarise all of them: what the random draws produced, how long gprMax took, and how close the predicted apex times are.

In [ ]:
scenes, runs = [], []
for folder in sorted(SCENES.glob("seed*"), key=lambda p: int(p.name[4:])):
    if (folder / "scene.json").exists() and (folder / "run.json").exists():
        scenes.append(json.loads((folder / "scene.json").read_text()))
        runs.append(json.loads((folder / "run.json").read_text()))
print(f"{len(scenes)} finished scenes")

targets = [t for s in scenes for t in s["targets"]]
apex = [r for run in runs for r in run.get("apex", []) if r["visible"]]
walls = [run["wall_s"] / 60 for run in runs if run.get("wall_s")]
errors = np.array([r["error_ns"] for r in apex])
quarter = apex[0]["quarter_period_ns"] if apex else 0.625
inside = np.abs(errors) <= quarter

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0, 0].hist([s["half_space"]["eps_r"] for s in scenes], bins=20, color="#b5651d")
axes[0, 0].set_title("soil half-space eps_r")
kinds = ["metal", "pvc", "air"]
axes[0, 1].bar(kinds, [sum(t["type"] == k for t in targets) for k in kinds], color=[TYPE_COLOUR[k] for k in kinds], edgecolor="k")
axes[0, 1].set_title("targets per type")
for k, colour in zip(kinds, ("#444444", "#1f77b4", "#2ca02c")):
    sub = [t for t in targets if t["type"] == k]
    axes[0, 2].scatter([t["radius_m"] * 100 for t in sub], [t["top_depth_m"] for t in sub], s=12, color=colour, label=k)
axes[0, 2].legend(fontsize=8)
axes[0, 2].invert_yaxis()
axes[0, 2].set_xlabel("radius [cm]")
axes[0, 2].set_ylabel("top depth [m]")
axes[0, 2].set_title("target size and depth")
axes[1, 0].hist([len(s["targets"]) for s in scenes], bins=np.arange(0.5, 4.5), rwidth=0.8)
axes[1, 0].set_title("targets per scene")
axes[1, 1].hist(walls, bins=20, color="0.5")
axes[1, 1].set_title(f"gprMax wall time per scene [min], mean {np.mean(walls):.1f}" if walls else "no timings")
axes[1, 2].hist(errors, bins=40, color="#2ca02c")
for sign in (-1, 1):
    axes[1, 2].axvline(sign * quarter, color="crimson", ls="--")
axes[1, 2].set_title(f"apex error [ns]: {inside.sum()} of {inside.size} within +/-{quarter:.2f}")
plt.tight_layout()
plt.show()

print(f"visible targets: {len(apex)} of {len(targets)}")
print(f"apex error: {inside.sum()} of {inside.size} within a quarter period; max |error| {np.abs(errors).max():.2f} ns; "
      f"{sum(r['on_window_edge'] for r in apex)} picks on the search-window edge")

### A gallery of processed B-scans

Six seeds after the processing chain, with the predicted hyperbolas. This is the material the next step turns into 224 x 224 training images with boxes.

In [ ]:
def processed(seed):
    folder = SCENES / f"seed{seed}"
    s = json.loads((folder / "scene.json").read_text())
    sc = loader.load_gprmax(folder / "scene_merged.h5")
    data, picks = proc.time_zero(sc.data, sc.dt)
    data = proc.dewow(data, sc.dt, dewow_ns)
    data = proc.remove_background(data)
    lo, hi = proc.default_band(s["frequency_hz"], 0.5 / sc.dt)
    data = proc.gain_agc(proc.bandpass(data, sc.dt, lo, hi), sc.dt, agc_ns)
    return s, sc, data, np.asarray(picks)

gallery = [s["seed"] for s in scenes][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, seed in zip(axes.flat, gallery):
    s, sc, data, picks = processed(seed)
    xs = np.asarray(s["ground_truth"]["targets"][0]["hyperbola_x_m"])
    t_ns = np.arange(data.shape[0]) * sc.dt * 1e9
    limit = np.percentile(np.abs(data), 99) or 1.0
    ax.imshow(data, extent=[xs[0], xs[-1], t_ns[-1], t_ns[0]], aspect="auto", cmap="gray", vmin=-limit, vmax=limit)
    for t in s["ground_truth"]["targets"]:
        ax.plot(xs, np.asarray(t["hyperbola_t_s"]) * 1e9 - picks[: xs.size] * sc.dt * 1e9, lw=0.8, ls="--")
    ax.set_xlim(xs[0], xs[-1])
    ax.set_title(f"seed {seed}: " + ", ".join(t["type"] for t in s["targets"]), fontsize=9)
    ax.set_ylim(t_ns[-1], 0)
for ax in axes[:, 0]:
    ax.set_ylabel("time after time zero [ns]")
plt.tight_layout()
plt.show()

## Summary

- The grid is fixed per preset and checked against the slowest velocity and the deepest target the preset allows.
- One seed gives one reproducible scene; `scene.json` keeps every parameter and the predicted hyperbola of every target.
- gprMax simulates one trace per antenna position (about 10 minutes per scene on 8 CPU threads at 400 MHz).
- The reference model removes the direct wave and the layers, which makes the apex check possible.
- Most visible targets fall within a quarter period of the prediction; the rest are picks on the search-window edge or second echoes of air-filled targets.

Next: `05a_render_synthetic.py` turns these B-scans into 224 x 224 training images with boxes that follow the same rule as the hand labels.